# Comparando Regressão Logística, Árvore de Decisão, Random Forest e MLPClassifier

## `Titanic Dataset`

Neste notebook, comparamos quatro modelos de classificação:

- **Regressão Logística**
- **Árvore de Decisão**
- **Random Forest**
- **MLPClassifier**

O objetivo é prever a sobrevivência de passageiros do Titanic com base em variáveis como sexo, idade, classe, tarifa e quantidade de familiares a bordo.

A proposta didática é observar como cada modelo lida com o mesmo problema:

- a **Regressão Logística** como modelo linear de referência;
- a **Árvore de Decisão** como modelo baseado em regras;
- a **Random Forest** como um conjunto de várias árvores, treinadas com amostras e divisões aleatórias;
- a **MLPClassifier** como uma rede neural multicamadas, capaz de aprender relações não lineares entre as variáveis de entrada e a classe prevista.


## Importação de libs

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)


## Carregamento dos dados

In [ ]:
# Carregar o dataset Titanic diretamente do seaborn
df = sns.load_dataset('titanic')

# Visualizar as primeiras linhas
df.head()

In [ ]:
df['survived'].value_counts(normalize=True)

In [ ]:
df.columns

## Seleção de features

Para simplificação, vamos **descartar variáveis redundantes** ou com muitos valores ausentes neste primeiro modelo (`deck`, `who`, `adult_male`, `embark_town`, `class`, `alive`).

In [ ]:
# Variáveis numéricas
num_features = ["age", "sibsp", "parch", "fare"]

# Variáveis categóricas
cat_features = ["pclass", "sex", "embarked", "alone"]

print("Variáveis numéricas:", num_features)
print("Variáveis categóricas:", cat_features)

# Criar novo DataFrame apenas com as variáveis escolhidas
df_model = df[num_features + cat_features].copy()

# Exibe amostra
df_model.head()

## Divisão em treino e teste

Para avaliar o modelo de forma justa, separamos os dados em **treino** e **teste**.  
- Usaremos `train_test_split` com **estratificação por `survived`** para manter a proporção de classes nos dois conjuntos.  
- Definimos um `random_state` para reprodutibilidade.  
- Mantemos o `df_model` apenas com as **features** e usamos `y = df["survived"]` como **target**.

In [ ]:
# X = features selecionadas; y = alvo (survived)
X = df_model.copy()
y = df["survived"].astype(int)

# Split estratificado (ex.: 75% treino, 25% teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Checagem rápida dos tamanhos e balanceamento
# Exibir resultados de forma mais clara
print("=== Divisão dos Dados ===")
print(f"Tamanho do treino: {X_train.shape[0]} amostras, {X_train.shape[1]} variáveis")
print(f"Tamanho do teste : {X_test.shape[0]} amostras, {X_test.shape[1]} variáveis\n")

print("=== Proporção de classes no treino ===")
for cls, prop in y_train.value_counts(normalize=True).round(3).items():
    print(f"Classe {cls}: {prop:.1%}")

print("\n=== Proporção de classes no teste ===")
for cls, prop in y_test.value_counts(normalize=True).round(3).items():
    print(f"Classe {cls}: {prop:.1%}")

In [ ]:
X_train.head()

### Treinamento dos modelos com Pipeline

Para treinar os classificadores, vamos utilizar **Regressão Logística**, **Árvore de Decisão**, **Random Forest** e **MLPClassifier**.

Antes do treinamento, precisamos transformar os dados para que os modelos consigam utilizá-los corretamente:

> **Regressão Logística:**
> - usa padronização nas variáveis numéricas, pois o modelo é sensível à escala dos dados;
> - usa codificação *One-Hot Encoding* nas variáveis categóricas.

> **MLPClassifier:**
> - também usa padronização nas variáveis numéricas, pois redes neurais são sensíveis à escala dos atributos;
> - precisa receber todas as variáveis em formato numérico, por isso também utiliza *One-Hot Encoding* nas variáveis categóricas;

> **Árvore de Decisão e Random Forest:**
> - não precisam de padronização das variáveis numéricas;
> - também precisam transformar variáveis categóricas em valores numéricos;

A ideia é manter o mesmo conjunto de treino e teste para todos os modelos, tornando a comparação mais justa.


In [ ]:
# Pré-processamento numérico: imputação de medianas
num_imputer = SimpleImputer(strategy="median")

# Pré-processamento numérico para Regressão Logística
num_transformer_reg = Pipeline(steps=[
    ("imputer", num_imputer),
    ("scaler", StandardScaler())
])

# Pré-processamento numérico para Árvore de Decisão
num_transformer_tree = Pipeline(steps=[
    ("imputer", num_imputer)
])

In [ ]:
# Pré-processamento categórico: imputação do mais frequente + OneHot
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
# ColumnTransformer para a Regressão Logística
preprocessor_reg = ColumnTransformer(
    transformers=[
        ("num", num_transformer_reg, num_features),
        ("cat", cat_transformer, cat_features)
    ]
)

# Modelo: Regressão Logística
logreg = LogisticRegression(max_iter=1000, random_state=42)

# Pipeline completo: pré-processamento + modelo
log_model = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("classifier", logreg)
])

log_model.fit(X_train, y_train)

In [ ]:
# ColumnTransformer para a Árvore de Decisão
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", num_transformer_tree, num_features),
        ("cat", cat_transformer, cat_features)
    ]
)

# Modelo: Árvore de Decisão
tree = DecisionTreeClassifier(random_state=42)

# Pipeline completo: pré-processamento + modelo
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("classifier", tree)
])

tree_model.fit(X_train, y_train)

### Modelo 3: Random Forest

A **Random Forest** combina várias árvores de decisão. Cada árvore é treinada com uma amostra aleatória dos dados, criada com reposição, e considera subconjuntos aleatórios de variáveis durante as divisões.

Neste exemplo, vamos usar:

- `n_estimators=100`: quantidade de árvores na floresta;
- `max_depth=3`: limita a profundidade das árvores, reduzindo o risco de overfitting;
- `oob_score=True`: calcula uma estimativa de desempenho usando amostras *out-of-bag*;
- `random_state=42`: garante reprodutibilidade.


In [ ]:
# Modelo: Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=3,
    min_samples_split=10,
    min_samples_leaf=10,
    max_features="sqrt",
    random_state=42,
    oob_score=True,
    n_jobs=-1
)

# Pipeline completo: pré-processamento + modelo
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("classifier", rf)
])

rf_model.fit(X_train, y_train)

In [ ]:
print("Modelo Random Forest treinado.")
print("OOB score:", round(rf_model.named_steps["classifier"].oob_score_, 3))

### Modelo 4: MLPClassifier

O **MLPClassifier** é uma rede neural artificial do tipo *Multilayer Perceptron*. Diferentemente dos modelos baseados em árvores, ele aprende combinações entre as variáveis por meio de camadas de neurônios artificiais.

Neste exemplo, podemos observar especialmente:

- `hidden_layer_sizes`: define a quantidade de neurônios nas camadas ocultas;
- `activation`: define a função de ativação usada pelos neurônios;
- `max_iter`: define o número máximo de iterações durante o treinamento;
- `random_state=42`: garante reprodutibilidade.


In [ ]:
# a rede também é sensível à escala —
# reaproveitamos o pré-processador da Regressão Logística
mlp = MLPClassifier(
    hidden_layer_sizes=(16, 8),  # 2 camadas ocultas
    activation="relu",
    max_iter=500,
    random_state=42,
)

mlp_model = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),  # scaler + one-hot
    ("classifier", mlp),
])

mlp_model.fit(X_train, y_train)
print("Modelo MLP treinado.")

## Avaliação dos modelos

Depois do treinamento, avaliamos o desempenho dos classificadores nos **dados de teste**.

Nesta primeira avaliação, usamos o limiar padrão de `0.5` para decidir se o passageiro sobreviveu (`1`) ou não sobreviveu (`0`).

Métricas principais:

- **Acurácia**: proporção total de acertos;
- **Precisão**: entre os casos previstos como sobreviventes, quantos realmente sobreviveram;
- **Recall**: entre os passageiros que realmente sobreviveram, quantos o modelo conseguiu identificar;
- **F1-score**: média harmônica entre precisão e recall.


In [ ]:
# Previsões no conjunto de teste
y_pred_log = log_model.predict(X_test)
y_pred_tree = tree_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)
y_pred_mlp = mlp_model.predict(X_test)

In [ ]:
# Função auxiliar para calcular as principais métricas

def avaliar_modelo(nome, y_true, y_pred):
    return {
        "Modelo": nome,
        "Acurácia": accuracy_score(y_true, y_pred),
        "Precisão": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred)
    }

resultados = pd.DataFrame([
    avaliar_modelo("Regressão Logística", y_test, y_pred_log),
    avaliar_modelo("Árvore de Decisão", y_test, y_pred_tree),
    avaliar_modelo("Random Forest", y_test, y_pred_rf),
    avaliar_modelo("MLP", y_test, y_pred_mlp)
])

resultados_formatados = resultados.copy()
for col in ["Acurácia", "Precisão", "Recall", "F1-score"]:
    resultados_formatados[col] = resultados_formatados[col].round(3)

resultados_formatados


### Relatórios de classificação

A seguir, observamos o relatório completo para cada modelo. Ele permite analisar o desempenho por classe:

- classe `0`: passageiro não sobreviveu;
- classe `1`: passageiro sobreviveu.


In [ ]:
print("=== Regressão Logística ===")
print(classification_report(y_test, y_pred_log, digits=3))

print("\n=== Árvore de Decisão ===")
print(classification_report(y_test, y_pred_tree, digits=3))

print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, digits=3))

print("\n=== MLP ===")
print(classification_report(y_test, y_pred_mlp, digits=3))


### Visualização comparativa das métricas

Para facilitar a leitura, vamos separar as métricas em pequenos gráficos horizontais. Em cada gráfico, o melhor modelo da métrica fica destacado em azul, enquanto os demais aparecem em cinza.


In [ ]:
metricas = ["Acurácia", "Precisão", "Recall", "F1-score"]

fig, axes = plt.subplots(1, len(metricas), figsize=(16, 4), sharey=True)

for ax, metrica in zip(axes, metricas):
    dados = resultados[["Modelo", metrica]].sort_values(metrica, ascending=True)
    melhor_valor = dados[metrica].max()
    cores = ["#1f77b4" if valor == melhor_valor else "#d0d0d0" for valor in dados[metrica]]

    ax.barh(dados["Modelo"], dados[metrica], color=cores)
    ax.set_title(metrica)
    ax.set_xlim(0, 1)
    ax.grid(axis="x", linestyle="--", alpha=0.3)
    ax.set_xlabel("Valor")

    for i, valor in enumerate(dados[metrica]):
        ax.text(valor + 0.08, i, f"{valor:.3f}", va="center", fontsize=9)

    if ax != axes[0]:
        ax.set_ylabel("")

fig.suptitle("Comparação dos modelos por métrica", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()


## Análise comparativa

Com base nas métricas obtidas:

- **Random Forest** e **MLP** tiveram a maior acurácia, com `0.789`;
- **MLP** teve a maior precisão, com `0.800`, ou seja, errou menos quando previu sobrevivência;
- **Regressão Logística** teve o melhor recall, com `0.698`, identificando mais passageiros sobreviventes;
- **Regressão Logística** também teve o melhor F1-score, com `0.714`, mostrando o melhor equilíbrio entre precisão e recall;
- **Árvore de Decisão** apresentou os menores resultados gerais entre os quatro modelos.

Assim, neste experimento, a **Regressão Logística** foi o modelo mais equilibrado. A **Random Forest** e o **MLP** tiveram boa acurácia, mas perderam em recall, principalmente o MLP. A **Árvore de Decisão** foi a alternativa mais simples, porém com desempenho inferior.


## Experimento com Grid Search para MLP

Agora vamos testar se o **MLPClassifier** melhora com ajuste de hiperparâmetros.

O objetivo é buscar uma combinação melhor para a rede neural, usando validação cruzada no conjunto de treino. Como o MLP teve precisão alta, mas recall menor, vamos otimizar pelo **F1-score**, que equilibra precisão e recall.


In [ ]:
from sklearn.model_selection import GridSearchCV


### Definição da grade de hiperparâmetros

A grade abaixo testa diferentes tamanhos de camada oculta, funções de ativação, valores de regularização e taxas iniciais de aprendizado.


In [ ]:
mlp_grid = MLPClassifier(
    max_iter=1000,
    early_stopping=True,
    random_state=42
)

mlp_grid_model = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("classifier", mlp_grid)
])

param_grid_mlp = {
    "classifier__hidden_layer_sizes": [(20,), (50,), (50, 25)],
    "classifier__activation": ["relu", "tanh"],
    "classifier__alpha": [0.0001, 0.001, 0.01],
    "classifier__learning_rate_init": [0.001, 0.01]
}


### Execução do Grid Search

O `GridSearchCV` treina várias versões do MLP e escolhe a melhor combinação com base no F1-score médio da validação cruzada.


In [ ]:
grid_search_mlp = GridSearchCV(
    estimator=mlp_grid_model,
    param_grid=param_grid_mlp,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search_mlp.fit(X_train, y_train)


### Melhor configuração encontrada


In [ ]:
print("Melhores hiperparâmetros:")
print(grid_search_mlp.best_params_)

print("\nMelhor F1-score médio na validação cruzada:")
print(round(grid_search_mlp.best_score_, 3))


### Avaliação no conjunto de teste

Depois de encontrar a melhor combinação, avaliamos o modelo ajustado nos mesmos dados de teste usados anteriormente.


In [ ]:
best_mlp_model = grid_search_mlp.best_estimator_
y_pred_mlp_grid = best_mlp_model.predict(X_test)

resultado_mlp_grid = pd.DataFrame([
    avaliar_modelo("MLP original", y_test, y_pred_mlp),
    avaliar_modelo("MLP Grid Search", y_test, y_pred_mlp_grid)
])

resultado_mlp_grid_formatado = resultado_mlp_grid.copy()
for col in ["Acurácia", "Precisão", "Recall", "F1-score"]:
    resultado_mlp_grid_formatado[col] = resultado_mlp_grid_formatado[col].round(3)

resultado_mlp_grid_formatado


### Relatório do MLP ajustado


In [ ]:
print(classification_report(y_test, y_pred_mlp_grid, digits=3))


## Conclusão

Neste notebook, comparamos quatro abordagens para classificação binária usando o mesmo conjunto de dados, a mesma divisão de treino e teste e o mesmo pré-processamento básico.

A principal conclusão didática é que:

- a **Regressão Logística** foi o modelo mais equilibrado nas métricas iniciais;
- a **Árvore de Decisão** é interpretável, mas teve desempenho inferior;
- a **Random Forest** combinou boa acurácia com maior robustez que uma árvore isolada;
- o **MLPClassifier** teve boa precisão, mas precisa de ajuste para tentar melhorar recall e F1-score.

Por isso, o experimento com **Grid Search** permite verificar se uma configuração melhor do MLP consegue superar o desempenho inicial.
